# Lecture 8: Integer Optimization

[![Open In Colab](images/colab-badge.svg)](https://colab.research.google.com/github/UvA-SSO/simopt-lecture-notes/blob/main/notebooks/lecture8_integer-optimization.ipynb)

An integer linear optimization (ILO) problem is an LO problem with the extra requirement
that some or all decision variables take integer values, $x_i \in \{0, 1, 2, \dots\}$, or
are binary, $x_i \in \{0, 1\}$. (A binary variable is just an integer one with the added
constraint $x_i \le 1$.)

In pulp this is a one-word change: set a variable's `cat` to `"Integer"` or `"Binary"`
instead of the default `"Continuous"`. Everything else about building and solving the
model is the same. Solving it, however, is a different matter: in general ILO is much
harder than LO, as this notebook and [Complexity](lecture11_complexity.ipynb) explain.

**Learning outcomes**

On completion of this notebook, you will be able to:

- recognize when a problem needs integer or binary decision variables, and formulate and
  solve it in pulp;
- explain why ILO is generally harder to solve than LO;
- explain how branch and bound finds the optimum of an ILO problem;
- write any LO problem in the general matrix form, and explain why linearity (but not
  integrality) is essential for efficient solvability.

In [ ]:
try:
    import pulp
except ModuleNotFoundError:  # pulp is not preinstalled on Google Colab
    import subprocess
    import sys

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pulp"], check=True)
    import pulp

## Why Integer Problems Are Harder

Take the product-mix problem from [Linear Optimization](lecture8_linear-optimization.ipynb)
and require whole units of $x$ and $y$. Its LO optimum was $(x, y) = (5, 2.5)$ with profit
17.5, not integer. Two things go wrong compared to LO:

- **the optimal corner is no longer feasible**, so the simplex reasoning ("the optimum is
  at a corner") does not directly help;
- **rounding the LO optimum is not enough**: rounding $y$ up to 3 while keeping $x = 5$
  violates the resource-2 constraint ($5 + 6 = 11 > 10$), and $(5, 2)$ or $(4, 3)$ each
  need checking. Evaluating the corners tells us little, because the integer optimum sits
  somewhere *inside* the feasible region.

Let pulp solve the integer version:

In [ ]:
profit = {"x": 2, "y": 3}
use = {"resource 1": {"x": 1, "y": 0}, "resource 2": {"x": 1, "y": 2}}
available = {"resource 1": 5, "resource 2": 10}
products = list(profit)

int_mix = pulp.LpProblem(name="integer_product_mix", sense=pulp.LpMaximize)
q = {p: pulp.LpVariable(name=p, lowBound=0, cat="Integer") for p in products}
int_mix += pulp.lpSum(profit[p] * q[p] for p in products)
for r, cap in available.items():
    int_mix += pulp.lpSum(use[r][p] * q[p] for p in products) <= cap
int_mix.solve(pulp.PULP_CBC_CMD(msg=False))
print("integer optimum:", {p: q[p].value() for p in products}, "profit", int_mix.objective.value())

:::{exercise}
:label: ex-6-9

Solve the larger LO problem from [Linear Optimization](lecture8_linear-optimization.ipynb)
again, once requiring all variables integer and once requiring them binary. Compare the
optimal objective values with the continuous one.
:::

## Branch and Bound

The method used to solve ILO problems exactly is branch and bound. It rests on two ideas,
stated here for a maximization problem:

1. The LO relaxation, the same problem with the integer constraints dropped
   ($x_i \in \{0,1\}$ becomes $0 \le x_i \le 1$; $x_i \in \{0,1,2,\dots\}$ becomes
   $x_i \ge 0$), is less restrictive, so its optimal value is an *upper bound (UB)* on the
   ILO optimum.
2. Any feasible *integer* solution gives a *lower bound (LB)* on the ILO optimum.

If a subproblem's UB is $\le$ the best LB found so far, that subproblem cannot contain a
better solution and is eliminated: this is what makes the method cleverer than checking
every integer point. When a relaxation is non-integer, we branch: pick a fractional
variable, say $x_j = 2.5$, and create two subproblems, one with $x_j \le 2$ and one with
$x_j \ge 3$. When a relaxation is already integer, it is a candidate LB and we stop
branching that subproblem.

For the integer product-mix problem:

- **Root.** LO relaxation optimum $(5, 2.5)$, value $17.5$, so UB $= 17.5$. Branch on the
  fractional $y = 2.5$.
- **Branch $y \le 2$.** Relaxation optimum $(5, 2)$, value $16$, integer, so LB $= 16$.
- **Branch $y \ge 3$.** Relaxation optimum $(4, 3)$, value $17$, integer, so LB $= 17$.

The best LB is $17$ from the right branch; the left branch's value $16$ is below it, so it
is eliminated. Every subproblem is now resolved, and $(4, 3)$ with profit $17$ is the
proven ILO optimum, matching what pulp reported above. Only three linear relaxations had
to be solved.

Many LO solvers handle integer constraints this way; the best (proprietary) ones for
large instances are CPLEX and Gurobi.

## The Knapsack Problem

The archetypal binary ILO problem is the knapsack problem: from a set of items, each with
a *reward* and a *weight*, choose a subset of maximum total reward whose total weight fits
a capacity. Applications include which items to load in a truck, cutting stock in a steel
plant, and simple forms of portfolio selection.

Consider capacity 10 and six items:

| | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| reward | 10 | 13 | 18 | 31 | 7 | 15 |
| weight | 2 | 3 | 4 | 7 | 1 | 3 |

With binary $x_i$ (1 = take item $i$):

$$
\begin{aligned}
\text{maximize} \quad & \sum_i r_i x_i \\
\text{subject to} \quad & \sum_i w_i x_i \le 10 \\
& x_i \in \{0, 1\} \text{ for all } i.
\end{aligned}
$$

In [ ]:
reward = [10, 13, 18, 31, 7, 15]
weight = [2, 3, 4, 7, 1, 3]
capacity = 10
n_items = len(reward)

knapsack = pulp.LpProblem(name="knapsack", sense=pulp.LpMaximize)
take = [pulp.LpVariable(name=f"x_{i + 1}", cat="Binary") for i in range(n_items)]
knapsack += pulp.lpSum(reward[i] * take[i] for i in range(n_items))
knapsack += pulp.lpSum(weight[i] * take[i] for i in range(n_items)) <= capacity
knapsack.solve(pulp.PULP_CBC_CMD(msg=False))
print("take items:", [i + 1 for i in range(n_items) if take[i].value() == 1])
print("total reward:", knapsack.objective.value())

The `build_knapsack` pattern and this data reappear in
[Modeling Tools and Solvers](lecture10_modeling-tools.ipynb) when we look at warm starting.

:::{exercise}
:label: ex-6-10

Take the knapsack solution pulp found above. Verify by hand that no single swap (adding
one currently-excluded item and removing whatever is needed to stay within capacity)
improves the total reward.
:::

:::{exercise}
:label: ex-6-11

Solve, by branch and bound *on paper*, the knapsack problem with rewards $(15, 9, 10, 5)$,
weights $(1, 3, 5, 4)$ and capacity 8. For the relaxation bound, fill the capacity with
items in decreasing order of reward-to-weight ratio, allowing a fraction of the last one.
Check your answer with pulp.
:::

## The General Formulation and Linearity

Having now seen both LO and ILO in action, we can step back and write down the general
form both fit into. An LO problem with $n$ decision variables and $m$ constraints is

$$
\begin{aligned}
\text{maximize} \quad & \sum_{j=1}^{n} p_j x_j \\
\text{subject to} \quad & \sum_{j=1}^{n} a_{ij} x_j \le b_i, \quad i = 1, \dots, m \\
& x_1, \dots, x_n \ge 0,
\end{aligned}
$$

or in matrix notation $\max\{p^T x \mid A x \le b,\ x \ge 0\}$, with $p, x$ column vectors
of length $n$, $b$ a column vector of length $m$, and $A$ an $m \times n$ matrix.

This one form covers more than it seems. Every other case can be rewritten into it:

- **minimization**: $\min p^T x = -\max (-p^T x)$;
- **"$\ge$" constraints**: $Ax \ge b \Leftrightarrow -Ax \le -b$;
- **"$=$" constraints**: $Ax = b \Leftrightarrow Ax \le b$ and $Ax \ge b$;
- **free (unrestricted) variables**: replace $x$ by $x^+ - x^-$ with $x^+, x^- \ge 0$.

What *cannot* be relaxed is linearity. If the objective is nonlinear, the optimum need
not lie at a corner (think of $\max -x^2$ on $[-1, 1]$, optimal at the interior point 0).
If a constraint is nonlinear, the feasible region is no longer a convex polyhedron, so it
can have several local optima and the simplex reasoning from
[Linear Optimization](lecture8_linear-optimization.ipynb) breaks down: you are not sure
you have found the best solution until you have checked every local optimum, which is
usually intractable. Nonlinear optimization therefore needs different, less efficient
algorithms.

Integrality, the one structured deviation from pure linearity that this notebook has been
about, sits in between: it does make a problem harder to solve (as branch and bound's
extra work above shows), but not intractably so, which is exactly why ILO is worth
treating as its own class rather than lumping it in with general nonlinear optimization.

## References

- Koole, G. (2019). *An Introduction to Business Analytics*. §6.4 "Integer Problems."